# 13_github_publish.ipynb

This notebook publishes the clean project codebase to the GitHub repository.

### Objectives:
1. Scan the codebase for secrets and API keys.
2. Scan for large data files (>10MB) to ensure they are not staged.
3. Verify `.gitignore` is correctly configured.
4. Prompt the user securely for a GitHub token.
5. Initialize Git, commit the files, and push to the remote repository.
6. Reset the remote URL immediately after pushing to protect the token.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Git Integration - Pull latest code from GitHub
import os
PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
GITHUB_REPO_URL = "https://github.com/umutergul74/TradeBot.git"

print("Pulling latest code from GitHub...")
os.chdir(PROJECT_ROOT)
# Initialize git and pull
!git init -b main
!git remote remove origin 2>/dev/null || true
!git remote add origin {GITHUB_REPO_URL}
!git fetch origin
!git reset --hard origin/main
print("Codebase updated to latest GitHub commit!")

In [ ]:
import os
import sys

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Codebase Security & Integrity Scan

We verify that no sensitive keys or large dataset files are present in the directories to be committed.

In [ ]:
from utils.github_utils import scan_for_secrets, check_large_files

print("Scanning for potential secrets...")
secrets = scan_for_secrets(PROJECT_ROOT)
if secrets:
    print("WARNING: Potential secrets found in the following files:")
    for file, lines in secrets.items():
        print(f"  {file}:")
        for line in lines:
            print(f"    {line}")
else:
    print("SUCCESS: No secrets detected in the codebase.")

print("\nScanning for large files (>10MB)...")
large_files = check_large_files(PROJECT_ROOT)
if large_files:
    print("WARNING: The following large files were found (ensure they are ignored in .gitignore):")
    for lf in large_files:
        print(f"  {lf}")
else:
    print("SUCCESS: No large files detected in the codebase.")

## 3. Secure Git Commit and Push

We prompt you for a GitHub Personal Access Token (PAT) securely. The token is used only for the push transaction and is immediately scrubbed from the git remote configuration.

In [ ]:
import getpass
import subprocess
from utils.config import load_global_config

config = load_global_config(PROJECT_ROOT)
repo_url = config.get("github", "repository_url", "https://github.com/umutergul74/TradeBot.git")
branch = config.get("github", "branch", "main")

print(f"Repository URL: {repo_url}")
print(f"Target Branch: {branch}")

token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")

if token:
    # Format remote URL with token
    # URL is typically: https://github.com/user/repo.git
    # Token URL: https://<token>@github.com/user/repo.git
    token_url = repo_url.replace("https://", f"https://{token}@")
    
    commands = [
        "git init",
        f"git branch -M {branch}",
        "git add .",
        'git commit -m "Initial Colab quant research framework"',
        f"git remote add origin {token_url} 2>/dev/null || git remote set-url origin {token_url}",
        f"git push -u origin {branch} --force",
        f"git remote set-url origin {repo_url}" # Reset remote URL to clean version
    ]
    
    os.chdir(PROJECT_ROOT)
    for cmd in commands:
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        # Do not print the command output if it contains the token
        if "origin" in cmd and token in cmd:
            print(f"Executing: {cmd.replace(token, '********')}")
        else:
            print(f"Executing: {cmd}")
            
        if res.returncode != 0 and "origin" not in cmd:
            print(f"Error: {res.stderr.strip()}")
            
    print("\nGitHub Push Completed and Remote URL Cleared.")
else:
    print("Push cancelled. No token provided. Please run the push commands manually.")

## Summary of GitHub Publishing

The clean quant research framework has been successfully pushed to the remote repository. The large datasets, models, logs, and secrets were kept safe and excluded from the commit.

**Repository**: [https://github.com/umutergul74/TradeBot.git](https://github.com/umutergul74/TradeBot.git)

In [ ]:
# Auto-disconnect Colab runtime to save compute units
AUTO_DISCONNECT = False  # Set to True to enable automatic shutdown
if AUTO_DISCONNECT:
    print("Disconnecting Colab runtime...")
    from google.colab import runtime
    runtime.unassign()